# LeetCode 457 — Circular Array Loop bằng Python

Link đề: https://leetcode.com/problems/circular-array-loop/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, Floyd Cycle Detection

Skeleton LeetCode:

```python
class Solution:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        ...
```

Bài này biến một mảng vòng thành **đồ thị mà mỗi đỉnh có đúng một cạnh đi ra** (functional graph). Tìm chu trình hợp lệ trong đồ thị đó là bài toán rất quen: Floyd "rùa và thỏ", đánh dấu đã thăm, hay thậm chí "bóc dần các đỉnh không nằm trong chu trình".

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng **vòng** `nums` (không có số 0); từ chỉ số `i` ta nhảy tới `(i + nums[i]) mod n`; tìm một **chu trình** có độ dài `k > 1` mà **mọi phần tử cùng dấu** (đều dương hoặc đều âm).
2. Nắm các công cụ Python: phép `%` cho chỉ số vòng (kể cả số âm), tích `a * b > 0` để kiểm tra **cùng dấu**, tập hợp/`dict` để nhớ đường đi, `collections.deque`, và cấu trúc **Union-Find (DSU)**.
3. Đi từ **mô phỏng `n` bước từ mỗi chỉ số `O(n^2)`** → **tô màu đường đi `O(n)` thời gian, `O(n)` bộ nhớ** → **Floyd `O(1)` bộ nhớ** → **bóc dần đỉnh bậc vào bằng 0 (Kahn)** → **Union-Find**.
4. Hiểu mấu chốt: mỗi chỉ số trỏ tới **đúng một** chỉ số kế, nên từ một điểm bắt đầu đường đi chỉ có một lối; rốt cuộc nó hoặc **vào một chu trình**, hoặc **dừng vì đổi dấu**. "Tự trỏ vào mình" (`k = 1`) và "đổi chiều" đều **không** tính.
5. Tránh các lỗi kinh điển: quên loại chu trình độ dài 1, quên kiểm tra cùng dấu, đánh dấu sai đỉnh đã thăm, và phá hỏng mảng đầu vào khi dùng đánh dấu tại chỗ.

## 2. Đề bài & ví dụ

Bạn chơi trò với một mảng **vòng** `nums` gồm các số nguyên **khác 0**. Đứng ở chỉ số `i`:

- Nếu `nums[i] > 0`: đi **tiến** `nums[i]` bước.
- Nếu `nums[i] < 0`: đi **lùi** `|nums[i]|` bước.

Mảng vòng nghĩa là tiến từ phần tử cuối thì về phần tử đầu, lùi từ phần tử đầu thì về phần tử cuối.

Một **chu trình** là dãy chỉ số `seq` độ dài `k` mà đi theo luật trên cho ra dãy lặp `seq[0] → seq[1] → … → seq[k-1] → seq[0] → …`, thoả:

- Mọi `nums[seq[j]]` đều **dương** hoặc đều **âm**.
- `k > 1`.

Trả về `true` nếu `nums` có chu trình, ngược lại `false`.

**Ví dụ 1**

- Input: `nums = [2,-1,1,2,2]`
- Output: `true`
- Giải thích: chu trình `0 → 2 → 3 → 0 → …`, các phần tử `2, 1, 2` đều dương.

**Ví dụ 2**

- Input: `nums = [-1,-2,-3,-4,-5,6]`
- Output: `false`
- Giải thích: chu trình duy nhất có độ dài 1 (chỉ số 5 → 5), không hợp lệ.

**Ví dụ 3**

- Input: `nums = [1,-1,5,1,4]`
- Output: `true`
- Giải thích: `0 → 1 → 0` có dấu hỗn hợp (không tính), nhưng `3 → 4 → 3` đều dương (độ dài 2) nên hợp lệ.

**Ràng buộc:**

- `1 <= nums.length <= 5000`
- `-1000 <= nums[i] <= 1000`
- `nums[i] != 0`

**Follow up:** giải trong `O(n)` thời gian và `O(1)` bộ nhớ phụ.

In [1]:
from typing import List
from collections import deque
import random
import time

examples = [([2, -1, 1, 2, 2], True), ([-1, -2, -3, -4, -5, 6], False), ([1, -1, 5, 1, 4], True)]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[2, -1, 1, 2, 2] -> kỳ vọng True
[-1, -2, -3, -4, -5, 6] -> kỳ vọng False
[1, -1, 5, 1, 4] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Chỉ số vòng bằng phép `%` (kể cả số âm)

Python có `%` luôn cho kết quả **không âm** khi chia cho số dương: `-1 % 5 == 4`. Vì thế chỉ số kế tiếp trong mảng vòng viết gọn là `(i + nums[i]) % n`, dù `nums[i]` âm hay lớn hơn `n` rất nhiều. (Trong C++/Java, `%` của số âm có thể âm nên phải `+ n` rồi `% n` lần nữa.)

In [2]:
n = 5
print(-1 % n, -6 % n, 7 % n)             # 4 4 2
nums = [2, -1, 1, 2, 2]
for i in range(len(nums)):
    nxt = (i + nums[i]) % len(nums)
    print(f"i={i}: nums[i]={nums[i]:>2} -> chỉ số kế {nxt}")

4 4 2
i=0: nums[i]= 2 -> chỉ số kế 2
i=1: nums[i]=-1 -> chỉ số kế 0
i=2: nums[i]= 1 -> chỉ số kế 3
i=3: nums[i]= 2 -> chỉ số kế 0
i=4: nums[i]= 2 -> chỉ số kế 1


### 3.2. Biến mảng thành "đồ thị mỗi đỉnh một cạnh ra"

Mỗi chỉ số `i` có **đúng một** chỉ số kế `next(i)`. Vẽ mũi tên `i → next(i)` ta được đồ thị mà mỗi đỉnh có **bậc ra bằng 1** (functional graph). Đi theo mũi tên từ bất kỳ đỉnh nào, vì chỉ có hữu hạn đỉnh nên sớm muộn ta **lặp lại một đỉnh** và đi vào một chu trình. Mọi "thành phần liên thông" của đồ thị có dạng **một chu trình + các nhánh cây đổ vào nó**.

In [3]:
nums = [2, -1, 1, 2, 2]
n = len(nums)
nxt = [(i + nums[i]) % n for i in range(n)]
print("next =", nxt)                     # [2, 0, 3, 0, 1]

start = 4
path = [start]
while True:
    start = nxt[start]
    print(" ->", start)
    if start in path:
        break
    path.append(start)
print("đường đi:", path, "| lặp lại ở", start)

next = [2, 0, 3, 0, 1]
 -> 1
 -> 0
 -> 2
 -> 3
 -> 0
đường đi: [4, 1, 0, 2, 3] | lặp lại ở 0


### 3.3. Kiểm tra "cùng dấu" bằng tích `a * b > 0`

Hai số nguyên khác 0 **cùng dấu** khi tích của chúng dương. Cách này gọn hơn `(a > 0) == (b > 0)` và dùng được khi so sánh nhiều lần. Nếu một phần tử đã bị đặt `0` (đánh dấu), tích bằng `0` nên bị coi "không hợp lệ".

In [4]:
print(3 * 5 > 0, -2 * -7 > 0, 3 * -1 > 0, 0 * 4 > 0)     # True True False False

True True False False


### 3.4. Chu trình độ dài 1 (tự trỏ vào mình)

Nếu `nums[i] % n == 0` (ví dụ `n = 5`, `nums[i] = 5` hay `-10`) thì `next(i) == i`: đi một vòng đầy đủ rồi về chính nó. Đó là chu trình độ dài `k = 1` — **đề không tính**.

In [5]:
nums = [-1, -2, -3, -4, -5, 6]
n = len(nums)
print([(i + nums[i]) % n for i in range(n)])    # [5, 5, 5, 5, 5, 5]: mọi đỉnh đổ vào đỉnh 5
# Với ví dụ 2: i=5, nums[5]=6, 6 % 6 == 0 -> trỏ vào chính nó
print((5 + nums[5]) % n)                        # 5

[5, 5, 5, 5, 5, 5]
5


### 3.5. Con trỏ rùa và thỏ (Floyd) trên một chuỗi `x → next(x)`

Hai con trỏ cùng xuất phát: `slow` đi **1** bước, `fast` đi **2** bước mỗi lượt. Nếu chuỗi vào một chu trình thì `fast` sẽ **đuổi kịp** `slow` trong chu trình (hai con trỏ gặp nhau). Nếu chuỗi kết thúc (ở đây: gặp phần tử đổi dấu) thì không bao giờ gặp. Ví dụ nhỏ với `next = [1, 2, 0, 3]` (đỉnh `3` tự trỏ):

In [6]:
nxt = [1, 2, 0, 3]
slow = fast = 0
for step in range(1, 6):
    slow = nxt[slow]
    fast = nxt[nxt[fast]]
    print(f"lượt {step}: slow={slow} fast={fast}", "<- gặp nhau" if slow == fast else "")
    if slow == fast:
        break

lượt 1: slow=1 fast=2 
lượt 2: slow=2 fast=1 
lượt 3: slow=0 fast=0 <- gặp nhau


### 3.6. Union-Find (DSU) mini

DSU gom các phần tử vào **nhóm**; `find(x)` trả về đại diện của nhóm chứa `x`, `union(a, b)` gộp hai nhóm. Nếu cố gộp hai phần tử **đã cùng nhóm** nghĩa là nối thêm cạnh đó tạo ra một **chu trình**. Ta dùng điều này ở Cách 5.

In [7]:
parent = list(range(5))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]       # nén đường đi
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False                         # đã cùng nhóm -> tạo chu trình
    parent[ra] = rb
    return True

print(union(0, 1), union(1, 2), union(0, 2))   # True True False (0,2 đã nối gián tiếp)

True True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [1, -1, 5, 1, 4]` (`n = 5`):

| `i` | `nums[i]` | `next(i) = (i + nums[i]) % 5` | cùng dấu với `i`? |
|---|---|---|---|
| 0 | 1 | 1 | `nums[0]=1`, `nums[1]=-1`: **khác dấu** |
| 1 | -1 | 0 | khác dấu |
| 2 | 5 | 2 | `next = chính nó` (tự trỏ) |
| 3 | 1 | 4 | `nums[4]=4` cùng dấu |
| 4 | 4 | 3 | `nums[3]=1` cùng dấu |

Đồ thị: `0 ⇄ 1` (khác dấu, loại), `2 ↺` (độ dài 1, loại), `3 ⇄ 4` (cùng dấu, độ dài 2 — **hợp lệ**) ⇒ `true`.

Quy trình tính tay: với mỗi chỉ số bắt đầu, **đi theo mũi tên**, dừng khi **đổi dấu**; nếu quay lại một đỉnh đã đi qua trong lượt này thì ta đã tìm thấy chu trình, kiểm tra độ dài `> 1`.

In [8]:
nums = [1, -1, 5, 1, 4]
n = len(nums)
for i in range(n):
    j = (i + nums[i]) % n
    print(f"{i} -> {j}",
          "| tự trỏ" if j == i else "",
          "| ĐỔI DẤU" if nums[i] * nums[j] < 0 else "")

0 -> 1  | ĐỔI DẤU
1 -> 0  | ĐỔI DẤU
2 -> 2 | tự trỏ 
3 -> 4  
4 -> 3  


## 5. Cách 1 — Brute Force / baseline: từ mỗi chỉ số, đi `n` bước

**Ý tưởng (nguyên lý chuồng bồ câu):** từ chỉ số `s`, đi liên tiếp theo mũi tên **chỉ khi vẫn cùng dấu với `nums[s]`**. Nếu đi được **`n` bước** mà không bị dừng thì ta đã thăm `n + 1` đỉnh trên đồ thị chỉ có `n` đỉnh ⇒ **chắc chắn đã vào chu trình**. Mọi đỉnh trên chu trình đều cùng dấu (vì ta chỉ bước qua đỉnh cùng dấu). Chu trình hợp lệ khi độ dài `> 1`; ta loại độ dài `1` bằng cách **không cho đi khi `next(i) == i`**.

Code `loop_walk_n`:

In [9]:
def loop_walk_n(nums):
    n = len(nums)
    for s in range(n):
        cur = s
        ok = True
        for _ in range(n):
            nx = (cur + nums[cur]) % n
            if nx == cur or nums[nx] * nums[s] < 0:      # tự trỏ hoặc đổi dấu -> dừng
                ok = False
                break
            cur = nx
        if ok:
            return True
    return False


for nums, expected in examples:
    print(nums, loop_walk_n(nums), "kỳ vọng", expected)

[2, -1, 1, 2, 2] True kỳ vọng True
[-1, -2, -3, -4, -5, 6] False kỳ vọng False
[1, -1, 5, 1, 4] True kỳ vọng True


**Độ phức tạp:** mỗi chỉ số bắt đầu tốn tối đa `n` bước ⇒ `Time = O(n^2)` (`5000^2 = 2.5·10^7`, chạy được nhưng chậm trong Python), `Space = O(1)`.

**Nút thắt:** các lượt đi từ những chỉ số khác nhau **đi lại cùng một đoạn đường** nhiều lần. Nếu một đỉnh đã được xác định "không dẫn tới chu trình hợp lệ", lần sau không cần xét lại.

## 6. Ý tưởng thuật toán / bước đột phá

Đồ thị chỉ có một lối ra từ mỗi đỉnh nên mỗi đỉnh **thuộc đúng một** "kết cục": hoặc dẫn vào một chu trình, hoặc bị chặn (đổi dấu / tự trỏ). Cho nên:

- **Mỗi đỉnh chỉ cần xử lý một lần.** Khi bắt đầu từ `s` và đi hết đường, **mọi đỉnh trên đường** đều có cùng kết cục: nếu tìm được chu trình hợp lệ ta dừng và trả `true`; nếu không, **đánh dấu cả đường là "đã xong, không có chu trình"** để lần sau không đi lại.
- Một đường chỉ đi qua các đỉnh **cùng dấu**, nên đỉnh dương chỉ xuất hiện trong lượt bắt đầu từ đỉnh dương và ngược lại; việc đánh dấu "đã xong" không làm lẫn lộn giữa hai dấu.

Từ ý tưởng này ta có ba hướng tối ưu: (a) **tô màu đường đi** (cần mảng trạng thái `O(n)`), (b) **Floyd + đánh dấu bằng cách đặt `0`** (không cần bộ nhớ thêm), (c) nhìn bài toán như tìm chu trình trong đồ thị `outdeg ≤ 1` (Kahn, DSU).

## 7. Các cách tiếp theo

### Cách 2 — Tô màu đường đi (visited toàn cục), `O(n)` thời gian

**Giải thích từng đoạn:**

1. Mảng `state[i]`: `0` chưa thăm, `1` đang nằm trên đường đi hiện tại, `2` đã xong.
2. Với mỗi `s` có `state[s] == 0`: bắt đầu đường đi mới, ghi lại các đỉnh vào `path`.
3. Từ `cur`, tính `nx`. **Dừng** nếu `nx` đổi dấu so với `nums[s]`. Nếu `state[nx] == 1` (đã nằm trên đường hiện tại): tìm thấy chu trình; độ dài `= len(path) - path.index(nx)`; nếu `> 1` ⇒ `True`. Nếu `state[nx] == 2`: đã xét, dừng.
4. Sau mỗi đường: đặt `state = 2` cho mọi đỉnh trong `path`.

In [10]:
def loop_coloring(nums):
    n = len(nums)
    state = [0] * n                   # 0 chưa thăm, 1 trên đường hiện tại, 2 đã xong
    for s in range(n):
        if state[s] != 0:
            continue
        path = []
        pos = {}                       # đỉnh -> vị trí trong path
        cur = s
        while True:
            state[cur] = 1
            pos[cur] = len(path)
            path.append(cur)
            nx = (cur + nums[cur]) % n
            if nums[nx] * nums[s] < 0:         # đổi dấu: dừng
                break
            if state[nx] == 1:                  # quay lại đỉnh trên đường hiện tại -> chu trình
                if len(path) - pos[nx] > 1:
                    return True
                break
            if state[nx] == 2:                  # đã xét xong ở lượt trước
                break
            cur = nx
        for v in path:
            state[v] = 2
    return False


for nums, expected in examples:
    print(nums, loop_coloring(nums), "kỳ vọng", expected)

[2, -1, 1, 2, 2] True kỳ vọng True
[-1, -2, -3, -4, -5, 6] False kỳ vọng False
[1, -1, 5, 1, 4] True kỳ vọng True


In [11]:
# Code nộp LeetCode — Cách 2
class Solution2:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)
        state = [0] * n
        for s in range(n):
            if state[s] != 0:
                continue
            path = []
            pos = {}
            cur = s
            while True:
                state[cur] = 1
                pos[cur] = len(path)
                path.append(cur)
                nx = (cur + nums[cur]) % n
                if nums[nx] * nums[s] < 0:
                    break
                if state[nx] == 1:
                    if len(path) - pos[nx] > 1:
                        return True
                    break
                if state[nx] == 2:
                    break
                cur = nx
            for v in path:
                state[v] = 2
        return False


print(Solution2().circularArrayLoop([2, -1, 1, 2, 2]), Solution2().circularArrayLoop([-1, -2, -3, -4, -5, 6]))

True False


**Độ phức tạp Cách 2:** mỗi đỉnh vào `path` đúng một lần ⇒ `Time = O(n)`, `Space = O(n)` (mảng `state`, `path`, `pos`).

> Lưu ý: khi `nx` **khác dấu** thì ta dừng **mà không thăm** `nx`; `nx` sẽ được xét riêng khi nó là điểm bắt đầu.

### Cách 3 — Floyd (rùa và thỏ) + đánh dấu bằng `0` (`O(1)` bộ nhớ)

**Ý tưởng:** làm đúng follow-up: không dùng bộ nhớ ngoài.

1. Với mỗi `i` còn `nums[i] != 0`: đặt `slow = i`, `fast = next(i)`.
2. Chừng nào `slow`, `fast`, `next(fast)` **cùng dấu** (tích `> 0`): nếu `slow == fast` thì đã gặp nhau trong chu trình; chu trình hợp lệ khi `slow != next(slow)` (độ dài `> 1`), rồi dừng. Ngược lại `slow` tiến 1, `fast` tiến 2.
3. Sau khi xét `i` mà không tìm thấy, **đặt `0`** cho mọi đỉnh trên đường đi từ `i` (cùng dấu): chúng không dẫn tới chu trình hợp lệ; về sau tích với `0` là `0` nên không đi vào nữa.

In [12]:
def loop_floyd(nums_in):
    nums = list(nums_in)                # sao chép để không phá input (bản nộp có thể sửa trực tiếp)
    n = len(nums)

    def nxt(i):
        return (i + nums[i]) % n

    for i in range(n):
        if nums[i] == 0:
            continue
        slow, fast = i, nxt(i)
        while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
            if slow == fast:
                if slow != nxt(slow):         # chu trình độ dài > 1
                    return True
                break                          # tự trỏ: bỏ
            slow = nxt(slow)
            fast = nxt(nxt(fast))
        j = i                                  # đánh dấu cả đường đi (cùng dấu) là đã xét
        while nums[j] * nums[nxt(j)] > 0:
            k = nxt(j)
            nums[j] = 0
            j = k
        nums[j] = 0                            # đỉnh cuối cùng của đường (tự trỏ hoặc trước chỗ đổi dấu)
    return False


for nums, expected in examples:
    print(nums, loop_floyd(nums), "kỳ vọng", expected)

[2, -1, 1, 2, 2] True kỳ vọng True
[-1, -2, -3, -4, -5, 6] False kỳ vọng False
[1, -1, 5, 1, 4] True kỳ vọng True


Một điểm tinh tế ở bước 3: vòng đánh dấu dừng ở đỉnh `j` mà đỉnh kế **khác dấu**, đã bị đặt `0`, hoặc là chính `j` (tự trỏ). Dòng `nums[j] = 0` cuối cùng đánh dấu nốt `j` (đỉnh này **cùng dấu** với đường đi). Ta **không** đặt `0` cho đỉnh kế khác dấu, vì đỉnh đó có thể thuộc một chu trình của chính nó (xem lỗi số 3 ở mục 13). Cũng chú ý: phải tính `k = nxt(j)` **trước** khi đặt `nums[j] = 0`, vì `nxt` đọc `nums[j]` (xem lỗi số 6).

In [13]:
# Code nộp LeetCode — Cách 3 (sửa mảng tại chỗ, O(1) bộ nhớ phụ)
class Solution3:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)

        def nxt(i):
            return (i + nums[i]) % n

        for i in range(n):
            if nums[i] == 0:
                continue
            slow, fast = i, nxt(i)
            while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
                if slow == fast:
                    if slow != nxt(slow):
                        return True
                    break
                slow = nxt(slow)
                fast = nxt(nxt(fast))
            j = i
            while nums[j] * nums[nxt(j)] > 0:
                k = nxt(j)
                nums[j] = 0
                j = k
            nums[j] = 0
        return False


print(Solution3().circularArrayLoop([1, -1, 5, 1, 4]), Solution3().circularArrayLoop([-1, -2, -3, -4, -5, 6]))

True False


**Độ phức tạp Cách 3:** mỗi đỉnh bị đặt `0` đúng một lần và Floyd chạy tuyến tính theo độ dài đường ⇒ `Time = O(n)`, `Space = O(1)` (không tính bản sao dùng để test).

### Cách 4 — "Bóc dần" đỉnh bậc vào bằng 0 (Kahn / sắp xếp tô-pô)

**Ý tưởng khác hẳn:** xây đồ thị chỉ gồm các cạnh **hợp lệ** `i → next(i)` (cùng dấu và `next(i) != i`). Mỗi đỉnh có **bậc ra ≤ 1**. Đỉnh **không nằm trên chu trình** thì có thể bị "bóc" dần: đỉnh có bậc vào `0` chắc chắn không thuộc chu trình; bóc nó đi sẽ giảm bậc vào của đỉnh kế. Lặp đến hết; **nếu còn đỉnh chưa bóc được** thì đó chính là các đỉnh trên chu trình (độ dài `> 1` vì không có tự trỏ).

**Giải thích từng đoạn:**

1. Tính `indeg[v]` = số cạnh hợp lệ đi vào `v`.
2. Đưa mọi đỉnh `indeg = 0` vào hàng đợi `deque`.
3. Lấy `u`, nếu có cạnh hợp lệ ra `u → v`: `indeg[v] -= 1`, nếu `0` thì cho `v` vào hàng đợi.
4. Đếm số đỉnh đã bóc. Nếu `< n` thì có chu trình.

In [14]:
def loop_kahn(nums):
    n = len(nums)
    out = [-1] * n                       # cạnh hợp lệ i -> out[i], hoặc -1 nếu không có
    indeg = [0] * n
    for i in range(n):
        j = (i + nums[i]) % n
        if j != i and nums[i] * nums[j] > 0:
            out[i] = j
            indeg[j] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    removed = 0
    while q:
        u = q.popleft()
        removed += 1
        v = out[u]
        if v != -1:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return removed < n


for nums, expected in examples:
    print(nums, loop_kahn(nums), "kỳ vọng", expected)

[2, -1, 1, 2, 2] True kỳ vọng True
[-1, -2, -3, -4, -5, 6] False kỳ vọng False
[1, -1, 5, 1, 4] True kỳ vọng True


In [15]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)
        out = [-1] * n
        indeg = [0] * n
        for i in range(n):
            j = (i + nums[i]) % n
            if j != i and nums[i] * nums[j] > 0:
                out[i] = j
                indeg[j] += 1
        q = deque(i for i in range(n) if indeg[i] == 0)
        removed = 0
        while q:
            u = q.popleft()
            removed += 1
            v = out[u]
            if v != -1:
                indeg[v] -= 1
                if indeg[v] == 0:
                    q.append(v)
        return removed < n


print(Solution4().circularArrayLoop([2, -1, 1, 2, 2]), Solution4().circularArrayLoop([-1, 2]))

True False


**Độ phức tạp Cách 4:** mỗi đỉnh vào hàng đợi tối đa một lần, mỗi cạnh xét một lần ⇒ `Time = O(n)`, `Space = O(n)`.

> Vì sao "còn đỉnh dư" là chu trình? Trong đồ thị bậc ra ≤ 1, một đỉnh không bóc được nghĩa là bậc vào của nó luôn `≥ 1` ngay cả sau khi bóc mọi tổ tiên không thuộc chu trình; đi **ngược** từ nó liên tục tìm được tổ tiên còn lại, mà chỉ có hữu hạn đỉnh nên phải **quay vòng** ⇒ chu trình.

### Cách 5 — Union-Find (gặp lại cùng một nhóm = chu trình)

**Ý tưởng:** cũng dùng đồ thị cạnh hợp lệ như Cách 4. Do bậc ra ≤ 1, một chu trình **vô hướng** trong đồ thị này bắt buộc là chu trình **có hướng**: mỗi đỉnh của chu trình vô hướng cần đúng hai cạnh kề trong chu trình, nhưng mỗi đỉnh chỉ có một cạnh **ra**, nên các cạnh phải xoay cùng chiều. Vậy: với mỗi cạnh hợp lệ `i → j`, nếu `find(i) == find(j)` trước khi gộp ⇒ có chu trình.

Trường hợp chu trình độ dài 2 (`i ⇄ j`): cạnh thứ nhất gộp `i, j`; cạnh thứ hai thấy cùng nhóm ⇒ phát hiện.

In [16]:
def loop_dsu(nums):
    n = len(nums)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i in range(n):
        j = (i + nums[i]) % n
        if j == i or nums[i] * nums[j] < 0:       # cạnh không hợp lệ
            continue
        ri, rj = find(i), find(j)
        if ri == rj:
            return True
        parent[ri] = rj
    return False


for nums, expected in examples:
    print(nums, loop_dsu(nums), "kỳ vọng", expected)

[2, -1, 1, 2, 2] True kỳ vọng True
[-1, -2, -3, -4, -5, 6] False kỳ vọng False
[1, -1, 5, 1, 4] True kỳ vọng True


In [17]:
# Code nộp LeetCode — Cách 5
class Solution5:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)
        parent = list(range(n))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        for i in range(n):
            j = (i + nums[i]) % n
            if j == i or nums[i] * nums[j] < 0:
                continue
            ri, rj = find(i), find(j)
            if ri == rj:
                return True
            parent[ri] = rj
        return False


print(Solution5().circularArrayLoop([1, -1, 5, 1, 4]), Solution5().circularArrayLoop([-2, 1, 2]))

True True


**Độ phức tạp Cách 5:** `Time = O(n · α(n)) ≈ O(n)`, `Space = O(n)`.

**Tóm tắt:** Cách 1 dễ hiểu nhất, chậm `O(n^2)`; Cách 2 tô màu là bản `O(n)` tự nhiên; Cách 3 đạt follow-up `O(1)` bộ nhớ; Cách 4, 5 nhìn bài như "có chu trình trong đồ thị bậc ra ≤ 1" và không cần mô phỏng đường đi.

## 8. TRACE từng bước

`trace_floyd` in từng lượt của rùa/thỏ cho mỗi điểm bắt đầu; `trace_kahn` in quá trình bóc đỉnh. Dùng ví dụ 1 `[2, -1, 1, 2, 2]` (có chu trình) và `[1, -1, 5, 1, 4]`.

In [18]:
def trace_floyd(nums_in):
    nums = list(nums_in)
    n = len(nums)
    nxt = lambda i: (i + nums[i]) % n
    print("nums =", nums)
    for i in range(n):
        if nums[i] == 0:
            print(f"--- start i={i}: đã bị đánh dấu 0, bỏ qua")
            continue
        slow, fast = i, nxt(i)
        print(f"--- start i={i}: slow={slow}, fast={fast}")
        while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
            if slow == fast:
                if slow != nxt(slow):
                    print(f"    slow == fast == {slow} và next(slow) = {nxt(slow)} != slow  => CHU TRÌNH HỢP LỆ")
                    return True
                print(f"    slow == fast == {slow} nhưng tự trỏ => bỏ")
                break
            slow = nxt(slow)
            fast = nxt(nxt(fast))
            print(f"    slow={slow}, fast={fast}")
        else:
            print("    dừng vì đổi dấu / gặp đỉnh 0")
        j = i
        marked = []
        while nums[j] * nums[nxt(j)] > 0:
            marked.append(j)
            k = nxt(j)
            nums[j] = 0
            j = k
        if j not in marked:
            marked.append(j)
        nums[j] = 0
        print("    đánh dấu 0:", marked, "-> nums =", nums)
    return False


print(trace_floyd([2, -1, 1, 2, 2]))

nums = [2, -1, 1, 2, 2]
--- start i=0: slow=0, fast=2
    slow=2, fast=0
    slow=3, fast=3
    slow == fast == 3 và next(slow) = 0 != slow  => CHU TRÌNH HỢP LỆ
True


In [19]:
print(trace_floyd([1, -1, 5, 1, 4]))

nums = [1, -1, 5, 1, 4]
--- start i=0: slow=0, fast=1
    dừng vì đổi dấu / gặp đỉnh 0
    đánh dấu 0: [0] -> nums = [0, -1, 5, 1, 4]
--- start i=1: slow=1, fast=0
    dừng vì đổi dấu / gặp đỉnh 0
    đánh dấu 0: [1] -> nums = [0, 0, 5, 1, 4]
--- start i=2: slow=2, fast=2
    slow == fast == 2 nhưng tự trỏ => bỏ
    đánh dấu 0: [2] -> nums = [0, 0, 0, 1, 4]
--- start i=3: slow=3, fast=4
    slow=4, fast=4
    slow == fast == 4 và next(slow) = 3 != slow  => CHU TRÌNH HỢP LỆ
True


Đọc bảng với `[2, -1, 1, 2, 2]` (`next = [2, 0, 3, 0, 1]`): xuất phát `i = 0` có `slow = 0`, `fast = 2`; sau một lượt `slow = 2`, `fast = next(next(2)) = next(3) = 0`; lượt hai `slow = 3`, `fast = next(next(0)) = next(2) = 3` ⇒ gặp nhau tại `3`; `next(3) = 0 != 3` ⇒ chu trình `0 → 2 → 3 → 0` có độ dài 3, cùng dấu dương ⇒ `True`.

In [20]:
def trace_kahn(nums):
    n = len(nums)
    out = [-1] * n
    indeg = [0] * n
    for i in range(n):
        j = (i + nums[i]) % n
        if j != i and nums[i] * nums[j] > 0:
            out[i] = j
            indeg[j] += 1
    print("cạnh hợp lệ out =", out)
    print("bậc vào indeg  =", indeg)
    q = deque(i for i in range(n) if indeg[i] == 0)
    removed = []
    while q:
        u = q.popleft()
        removed.append(u)
        v = out[u]
        msg = ""
        if v != -1:
            indeg[v] -= 1
            msg = f" -> indeg[{v}] = {indeg[v]}"
            if indeg[v] == 0:
                q.append(v)
                msg += " (đưa vào hàng đợi)"
        print(f"bóc {u}{msg}")
    left = [i for i in range(n) if i not in removed]
    print("đã bóc:", removed, "| còn lại (nằm trên chu trình):", left)
    return bool(left)


print(trace_kahn([1, -1, 5, 1, 4]))

cạnh hợp lệ out = [-1, -1, -1, 4, 3]
bậc vào indeg  = [0, 0, 0, 1, 1]
bóc 0
bóc 1
bóc 2
đã bóc: [0, 1, 2] | còn lại (nằm trên chu trình): [3, 4]
True


Kết quả: `0 ⇄ 1` khác dấu và `2` tự trỏ nên **không có cạnh hợp lệ**, cả ba đỉnh `0, 1, 2` có bậc vào `0` và bị bóc ngay; còn `3 ⇄ 4` bậc vào `1` mỗi đỉnh, không bóc được ⇒ chu trình hợp lệ.

## 9. Code nộp LeetCode

Bản nộp chính: Floyd + đánh dấu `0` (Cách 3). Nó đạt follow-up (`O(n)` thời gian, `O(1)` bộ nhớ phụ). Lưu ý: bản này **sửa mảng đầu vào**; hãy truyền bản sao khi test.

In [21]:
class Solution:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)

        def nxt(i):
            return (i + nums[i]) % n

        for i in range(n):
            if nums[i] == 0:
                continue
            slow, fast = i, nxt(i)
            while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
                if slow == fast:
                    if slow != nxt(slow):
                        return True
                    break
                slow = nxt(slow)
                fast = nxt(nxt(fast))
            j = i
            while nums[j] * nums[nxt(j)] > 0:
                k = nxt(j)
                nums[j] = 0
                j = k
            nums[j] = 0
        return False


for nums, expected in examples:
    assert Solution().circularArrayLoop(list(nums)) == expected
print("OK các ví dụ đề")

OK các ví dụ đề


## 10. Vì sao thuật toán đúng

**Mô hình.** Gọi `f(i) = (i + nums[i]) mod n`. Đồ thị `i → f(i)` có bậc ra đúng 1 nên mọi chu trình là **cách duy nhất** để lặp: từ một đỉnh, dãy `i, f(i), f(f(i)), …` hoặc kết thúc (nếu ta tự dừng vì đổi dấu) hoặc vào một chu trình duy nhất. Điều kiện đề: chu trình **cùng dấu** và **độ dài ≥ 2**.

**Cách 1.** Nếu từ `s` đi `n` bước liên tiếp mà không gặp đỉnh khác dấu và không gặp bước tự trỏ, ta đã thăm `n + 1` đỉnh nên (chuồng bồ câu) một đỉnh lặp lại — các đỉnh từ lần lặp ấy tạo chu trình, **toàn bộ chu trình nằm trong dãy đã đi** nên cùng dấu `nums[s]`; không có bước tự trỏ nên độ dài ≥ 2. Ngược lại nếu có chu trình hợp lệ, bắt đầu từ một đỉnh của nó, đi `n` bước đều hợp lệ. ⇒ đúng.

**Cách 2 (tô màu).** Mỗi đỉnh có kết cục duy nhất, nên khi đường đi gặp đỉnh `state = 2` (đã xét xong) ta bỏ qua an toàn; gặp `state = 1` nghĩa là quay lại chính đường này ⇒ chu trình (độ dài `len(path) - pos`). Đường bị dừng vì đổi dấu thì không đi tiếp; đỉnh đổi dấu **không** bị tô nên vẫn xét sau.

**Cách 3 (Floyd).** Trong chuỗi `i, f(i), …` toàn đỉnh cùng dấu, nếu có chu trình thì `slow` và `fast` gặp nhau trong chu trình (định lý Floyd: khi cả hai vào chu trình khoảng cách tương đối tăng 1 mỗi lượt theo modulo độ dài chu trình). Gặp nhau mà `f(slow) != slow` ⇒ chu trình độ dài ≥ 2. Nếu `f(slow) == slow` ⇒ chu trình tự trỏ, dẫn tới **chuỗi từ `i` chỉ tới tự trỏ**: không có chu trình hợp lệ ở chuỗi này (vì mỗi đỉnh chỉ có một lối). Nếu `slow`/`fast` gặp phần tử khác dấu hay `0` thì chuỗi cắt ngang trước khi vào chu trình. Đặt `0` cho đường đi là an toàn vì mọi đỉnh trên đó có cùng kết cục "không dẫn tới chu trình hợp lệ", và đỉnh khác dấu không bị động tới.

**Cách 4 (Kahn).** Chỉ giữ cạnh hợp lệ (cùng dấu, không tự trỏ). Bóc lặp đỉnh bậc vào 0: các đỉnh bị bóc không thuộc chu trình vì chu trình có mọi đỉnh bậc vào ≥ 1 mà bóc đỉnh ngoài chu trình không làm giảm bậc vào đỉnh trong chu trình (cạnh của chúng đi vào chu trình, chúng không đi từ chu trình). Phần còn lại sau khi bóc là hợp các chu trình ⇒ có đỉnh dư khi và chỉ khi có chu trình hợp lệ.

**Cách 5 (DSU).** Đồ thị cạnh hợp lệ có bậc ra ≤ 1. Một chu trình vô hướng trong đó có `k` đỉnh và `k` cạnh, mỗi đỉnh chỉ có 1 cạnh ra ⇒ mỗi đỉnh phát ra đúng một cạnh của chu trình ⇒ các cạnh cùng chiều, tức chu trình có hướng. Ngược lại mọi chu trình có hướng (độ dài ≥ 2) làm cạnh cuối cùng nối hai đỉnh đã cùng nhóm. ⇒ đúng.

Mục 12 đối chiếu cả 5 cách với một **oracle theo đúng định nghĩa** (đỉnh thuộc chu trình nếu đi vòng quay về chính nó) trên rất nhiều mảng nhỏ.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đi `n` bước từ mỗi chỉ số | `O(n^2)` | `O(1)` | `n = 5000` ⇒ `2.5·10^7` bước |
| 2. Tô màu đường đi | `O(n)` | `O(n)` | an toàn, dễ hiểu |
| 3. Floyd + đánh dấu `0` | `O(n)` | `O(1)` | **follow-up**; sửa input |
| 4. Kahn (bóc bậc vào 0) | `O(n)` | `O(n)` | không mô phỏng đường đi |
| 5. Union-Find | `O(n α(n))` | `O(n)` | rất ngắn gọn |

Với `n ≤ 5000`, mọi cách chạy được; Cách 1 chạy chậm nhất trong Python nhưng vẫn qua.

## 12. Bộ test

### 12.1. Oracle độc lập + ví dụ đề + edge case

**Oracle** làm đúng theo định nghĩa: một chỉ số `i` thuộc chu trình nếu đi theo `f` quay lại `i` sau `L` bước; chu trình hợp lệ khi `L > 1` và mọi đỉnh trong chu trình **cùng dấu**.

In [22]:
def cycle_nodes(nums):
    """Tập chỉ số nằm trên một chu trình hợp lệ (cùng dấu, độ dài >= 2) — theo đúng định nghĩa."""
    n = len(nums)
    res = set()
    for i in range(n):
        cur = i
        path = [i]
        for L in range(1, n + 1):
            cur = (cur + nums[cur]) % n
            if cur == i:
                if L > 1 and all(nums[p] * nums[i] > 0 for p in path):
                    res.update(path)
                break
            path.append(cur)
    return res


def oracle(nums):
    return len(cycle_nodes(nums)) > 0


solutions = {
    "walk_n": loop_walk_n,
    "coloring": loop_coloring,
    "floyd": loop_floyd,
    "kahn": loop_kahn,
    "dsu": loop_dsu,
    "Solution": lambda nums: Solution().circularArrayLoop(list(nums)),
}

edge = [
    ([2, -1, 1, 2, 2], True),
    ([-1, -2, -3, -4, -5, 6], False),
    ([1, -1, 5, 1, 4], True),
    ([1], False),                     # n = 1: tự trỏ
    ([-1], False),
    ([7], False),
    ([1, 1], True),                   # 0 -> 1 -> 0, cùng dấu
    ([-1, -1], True),
    ([1, -1], False),                 # chu trình khác dấu
    ([2, 2], False),                  # cả hai tự trỏ
    ([1, 2], False),                  # 0 -> 1 -> 1 (tự trỏ)
    ([-2, 1, 2], True),               # đường đi bắt đầu khác dấu rồi vào chu trình dương
    ([3, 1, 2], True),                # next = [0, 2, 1]: 0 tự trỏ, 1 <-> 2 cùng dấu
    ([1000, 1000, 1000, 1, 1], False),# bước lớn hơn n vẫn phải lấy modulo
    ([-1000, 1, 1, 1], False),
    ([2, -1, 1, 2, 2, -3, 1], False), # chu trình 2->3->5->2 lẫn dấu
]
for nums, expected in edge:
    assert oracle(nums) == expected, ("oracle", nums, oracle(nums))
    for name, fn in solutions.items():
        assert fn(nums) == expected, (name, nums, expected)
print("OK", len(edge), "ca, mọi cách đều khớp oracle và đáp án tính tay")

OK 16 ca, mọi cách đều khớp oracle và đáp án tính tay


Các ca sau ví dụ đề là ta tự dựng và tính tay. Ví dụ `[3, 1, 2]` (`n = 3`): `next = [0, 2, 1]`, `0` tự trỏ, còn `1 ⇄ 2` cùng dấu dương ⇒ **True**. Ví dụ `[1000, 1000, 1000, 1, 1]` (`n = 5`): `1000 % 5 == 0` nên ba phần tử đầu tự trỏ, còn `3 → 4 → 0 → 0` kết thúc ở tự trỏ ⇒ **False**.

### 12.2. Phản ví dụ cho các cách làm sai

Ba "lời giải nghe có lý" nhưng sai: (1) quên loại chu trình độ dài 1, (2) quên kiểm tra cùng dấu, (3) đánh dấu `0` cả đỉnh khác dấu.

In [23]:
def wrong_no_selfloop(nums):          # SAI: coi tự trỏ là chu trình
    n = len(nums)
    state = [0] * n
    for s in range(n):
        if state[s]:
            continue
        path, pos, cur = [], {}, s
        while True:
            state[cur] = 1
            pos[cur] = len(path)
            path.append(cur)
            nx = (cur + nums[cur]) % n
            if nums[nx] * nums[s] < 0:
                break
            if state[nx] == 1:
                return True            # thiếu kiểm tra độ dài > 1
            if state[nx] == 2:
                break
            cur = nx
        for v in path:
            state[v] = 2
    return False

def wrong_no_sign(nums):               # SAI: bỏ qua dấu
    n = len(nums)
    out = [(i + nums[i]) % n for i in range(n)]
    indeg = [0] * n
    for i in range(n):
        if out[i] != i:
            indeg[out[i]] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    removed = 0
    while q:
        u = q.popleft(); removed += 1
        v = out[u]
        if v != u:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return removed < n

def wrong_mark_other_sign(nums_in):     # SAI: đặt 0 luôn đỉnh khác dấu làm dừng đường
    nums = list(nums_in)
    n = len(nums)
    nxt = lambda i: (i + nums[i]) % n
    for i in range(n):
        if nums[i] == 0:
            continue
        slow, fast = i, nxt(i)
        while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
            if slow == fast:
                if slow != nxt(slow):
                    return True
                break
            slow = nxt(slow)
            fast = nxt(nxt(fast))
        j = i
        while True:
            k = nxt(j)
            s = nums[j]
            nums[j] = 0
            if s * nums[k] <= 0:
                nums[k] = 0            # SAI: xoá luôn đỉnh khác dấu
                break
            j = k
    return False

tests = {
    "quên tự trỏ": (wrong_no_selfloop, [2, 2]),
    "quên dấu": (wrong_no_sign, [1, -1]),
    "đặt 0 đỉnh khác dấu": (wrong_mark_other_sign, [-2, 1, 2]),
}
for name, (fn, nums) in tests.items():
    print(f"{name:22s} {nums} -> sai: {fn(nums)}, đúng: {oracle(nums)}")
    assert fn(nums) != oracle(nums)

quên tự trỏ            [2, 2] -> sai: True, đúng: False
quên dấu               [1, -1] -> sai: True, đúng: False
đặt 0 đỉnh khác dấu    [-2, 1, 2] -> sai: False, đúng: True


### 12.3. Đối chiếu ngẫu nhiên có seed (mảng nhỏ)

Sinh mảng `n ≤ 9`, giá trị khác 0 trong `[-12, 12]` (có cả bội của `n` để tạo tự trỏ và bước lớn hơn `n`). Đối chiếu cả 5 cách với oracle.

In [24]:
random.seed(457)
true_cnt = 0
for _ in range(6000):
    n = random.randint(1, 9)
    vals = [v for v in range(-12, 13) if v != 0]
    nums = [random.choice(vals) for _ in range(n)]
    exp = oracle(nums)
    true_cnt += exp
    for name, fn in solutions.items():
        assert fn(nums) == exp, (name, nums, exp)
print("OK 6000 mảng; số mảng có chu trình hợp lệ:", true_cnt)

OK 6000 mảng; số mảng có chu trình hợp lệ: 1287


### 12.4. Vét cạn mọi mảng nhỏ và mảng lớn

Mọi mảng độ dài `n ≤ 4` với giá trị trong `[-4, 4] \ {0}` (tổng `8 + 64 + 512 + 4096` mảng), rồi mảng `n = 5000` (so thời gian).

In [25]:
from itertools import product

vals = [v for v in range(-4, 5) if v != 0]
cnt = 0
for n in range(1, 5):
    for nums in product(vals, repeat=n):
        exp = oracle(nums)
        for name, fn in solutions.items():
            assert fn(nums) == exp, (name, nums, exp)
        cnt += 1
print("đã đối chiếu", cnt, "mảng nhỏ: khớp hoàn toàn")

đã đối chiếu 4680 mảng nhỏ: khớp hoàn toàn


In [26]:
random.seed(5000)
n = 5000
cases = {
    "ngẫu nhiên": [random.choice([v for v in range(-1000, 1001) if v != 0]) for _ in range(n)],
    "toàn +1 (chu trình dài n)": [1] * n,
    "toàn +n (toàn tự trỏ)": [n] * n,
    "dương nhưng ngắt bởi âm cuối": [1] * (n - 1) + [-1],
    "xen kẽ dấu": [(-1) ** i * (i % 7 + 1) for i in range(n)],
}
for name, arr in cases.items():
    res = {}
    for fname, fn in solutions.items():
        t0 = time.perf_counter()
        r = fn(arr)
        res[fname] = (r, round(time.perf_counter() - t0, 3))
    answers = {v[0] for v in res.values()}
    assert len(answers) == 1, (name, res)
    print(f"{name:30s} -> {answers.pop()}   thời gian:", {k: v[1] for k, v in res.items()})

ngẫu nhiên                     -> False   thời gian: {'walk_n': 0.002, 'coloring': 0.001, 'floyd': 0.001, 'kahn': 0.001, 'dsu': 0.001, 'Solution': 0.001}
toàn +1 (chu trình dài n)      -> True   thời gian: {'walk_n': 0.0, 'coloring': 0.001, 'floyd': 0.001, 'kahn': 0.001, 'dsu': 0.001, 'Solution': 0.002}
toàn +n (toàn tự trỏ)          -> False   thời gian: {'walk_n': 0.001, 'coloring': 0.001, 'floyd': 0.003, 'kahn': 0.001, 'dsu': 0.0, 'Solution': 0.003}


dương nhưng ngắt bởi âm cuối   -> False   thời gian: {'walk_n': 1.088, 'coloring': 0.001, 'floyd': 0.001, 'kahn': 0.001, 'dsu': 0.001, 'Solution': 0.002}
xen kẽ dấu                     -> False   thời gian: {'walk_n': 0.001, 'coloring': 0.001, 'floyd': 0.001, 'kahn': 0.001, 'dsu': 0.001, 'Solution': 0.001}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên loại chu trình độ dài 1 (`k > 1`)

`nums[i] % n == 0` khiến `i` trỏ về chính nó. Đề **không** tính chu trình như vậy.

In [27]:
nums = [2, 2]                     # n = 2: cả hai phần tử tự trỏ
print("sai :", wrong_no_selfloop(nums))
print("đúng:", Solution().circularArrayLoop(list(nums)))

sai : True
đúng: False


**Đúng:** loại cạnh `i → i` (Cách 4/5) hoặc kiểm tra `slow != next(slow)` (Cách 3) hoặc `độ dài chu trình > 1` (Cách 2).

### Lỗi 2: Quên kiểm tra cùng dấu

`[1, -1]`: `0 → 1 → 0` là chu trình độ dài 2 nhưng **khác dấu** nên không hợp lệ.

In [28]:
nums = [1, -1]
print("sai :", wrong_no_sign(nums))
print("đúng:", Solution().circularArrayLoop(list(nums)))

sai : True
đúng: False


**Đúng:** chỉ đi qua / chỉ nối các đỉnh có `nums[i] * nums[next(i)] > 0`.

### Lỗi 3: Đánh dấu "đã xét" cả đỉnh khác dấu

Khi đường đi gặp một đỉnh khác dấu và dừng, đỉnh đó **chưa** được xác định kết cục; nếu đặt `0` luôn cho nó thì ta bỏ sót chu trình của chính nó.

In [29]:
nums = [-2, 1, 2]
# 0 (âm) -> 1 (dương) đổi dấu; 1 <-> 2 là chu trình dương hợp lệ
print("sai :", wrong_mark_other_sign(nums))
print("đúng:", Solution().circularArrayLoop(list(nums)))

sai : False
đúng: True


**Đúng:** khi đổi dấu, dừng **mà không đụng** đỉnh khác dấu (đoạn `nums[j] = 0` chỉ áp dụng cho các đỉnh cùng dấu trên đường).

### Lỗi 4: Phá hỏng mảng đầu vào (đánh dấu tại chỗ)

Bản Floyd + đặt `0` **sửa `nums`**. Nếu mảng không có chu trình hợp lệ, **mọi ô** bị đặt `0`: dữ liệu gốc mất. Nếu bạn dùng lại cùng list để chạy cách khác, đối chiếu hay in ra, bạn sẽ nhìn thấy toàn số 0.

In [30]:
nums2 = [1, -1, 2, -3]
print("oracle:", oracle(nums2))           # False: không có chu trình hợp lệ
buf = list(nums2)
print(Solution().circularArrayLoop(buf), "-> mảng sau khi gọi:", buf)     # toàn 0
print("mảng gốc còn nguyên vì ta đã sao chép:", nums2)

oracle: False
False -> mảng sau khi gọi: [0, 0, 0, 0]
mảng gốc còn nguyên vì ta đã sao chép: [1, -1, 2, -3]


**Đúng:** truyền `list(nums)` (bản sao) khi cần dùng lại input (như các `assert` trong notebook này), hoặc dùng bản tô màu `state` (Cách 2) không đụng tới `nums`. Trên LeetCode việc sửa input là chấp nhận được vì mỗi test chạy riêng.

### Lỗi 5: Bước lớn hơn `n` hoặc số âm — quên `%`

`i + nums[i]` có thể vượt `n` hoặc âm; luôn lấy `% n`.

In [31]:
nums = [1000, 1000, 1000, 1, 1]
n = len(nums)
try:
    print(nums[0 + nums[0]])                    # quên %
except IndexError as e:
    print("IndexError:", e)
print("đúng:", nums[(0 + nums[0]) % n])

IndexError: list index out of range
đúng: 1000


### Lỗi 6: Đặt `0` rồi mới tính bước kế (`j = nxt(j)` đọc phải số 0)

`nxt(j) = (j + nums[j]) % n`. Nếu đã gán `nums[j] = 0` mà **sau đó** mới gọi `nxt(j)` thì được `(j + 0) % n = j`: con trỏ đứng yên, vòng đánh dấu dừng ngay và chỉ đánh dấu được **một** đỉnh. Kết quả vẫn đúng nhưng thuật toán thoái hoá thành `O(n^2)`, đúng loại lỗi mà test nhỏ không phát hiện.

In [32]:
def loop_floyd_variant(nums_in, fixed):
    nums = list(nums_in)
    n = len(nums)
    nxt = lambda i: (i + nums[i]) % n
    starts = 0                                  # đếm số lần phải chạy Floyd từ một điểm bắt đầu
    for i in range(n):
        if nums[i] == 0:
            continue
        starts += 1
        slow, fast = i, nxt(i)
        while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
            if slow == fast:
                if slow != nxt(slow):
                    return True, starts
                break
            slow, fast = nxt(slow), nxt(nxt(fast))
        j = i
        while nums[j] * nums[nxt(j)] > 0:
            if fixed:
                k = nxt(j)                     # ĐÚNG: tính bước kế trước khi đặt 0
                nums[j] = 0
                j = k
            else:
                nums[j] = 0
                j = nxt(j)                     # SAI: nxt(j) đọc nums[j] vừa bị đặt 0 -> j đứng yên
        nums[j] = 0
    return False, starts

arr = [1] * 2999 + [-1]
print("bản lỗi :", loop_floyd_variant(arr, fixed=False))   # phải chạy Floyd từ gần như mọi đỉnh
print("bản đúng:", loop_floyd_variant(arr, fixed=True))    # chỉ vài điểm bắt đầu

bản lỗi : (False, 3000)
bản đúng: (False, 2)


Bản đúng chỉ cần **hai** điểm bắt đầu (đỉnh `0` quét sạch toàn đường dương, đỉnh cuối xử lý nốt phần còn lại), bản lỗi phải khởi động Floyd từ hầu hết các đỉnh. **Đúng:** lưu `k = nxt(j)` rồi mới `nums[j] = 0; j = k`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"mỗi vị trí nhảy tới đúng một vị trí khác"** → đó là **functional graph**: nghĩ ngay tới **chu trình + rùa/thỏ** (Floyd).
- Cụm "cùng chiều" = ràng buộc **cùng dấu**; cụm "k > 1" = loại **tự trỏ**. Hai điều kiện này gần như là toàn bộ "bẫy" của bài.
- Follow-up `O(1)` bộ nhớ gợi ý **đánh dấu tại chỗ** (đặt `0`, vì đề cam kết không có số 0 ban đầu).
- Cùng họ: 141/142 (Linked List Cycle), 287 (Find the Duplicate Number), 202 (Happy Number), 565 (Array Nesting), 1306 (Jump Game III), 2127.
- Mẹo test: tạo mảng `[n] * n` (toàn tự trỏ) và `[1] * n` (một chu trình dài) để kiểm tra hai thái cực.

## 15. Biến thể của bài

1. **Trả về một chu trình hợp lệ** (danh sách chỉ số) thay vì `True/False`.
2. **Độ dài chu trình hợp lệ dài nhất.**
3. **Đếm số chỉ số nằm trên chu trình hợp lệ.**
4. **Không yêu cầu cùng dấu** (mọi chu trình độ dài `> 1`).
5. **Mảng không vòng** (nhảy ra ngoài là dừng) — đồ thị không có chu trình mới nếu... (Jump Game).

Dưới đây là biến thể 1–4 viết trên cơ sở `cycle_nodes` (định nghĩa theo đề), kèm test với bộ kiểm tra tính hợp lệ.

In [33]:
def find_valid_cycle(nums):
    """Trả về danh sách chỉ số của MỘT chu trình hợp lệ (theo thứ tự đi), hoặc None."""
    n = len(nums)
    for i in range(n):
        cur, path = i, [i]
        for L in range(1, n + 1):
            cur = (cur + nums[cur]) % n
            if cur == i:
                if L > 1 and all(nums[p] * nums[i] > 0 for p in path):
                    return path
                break
            path.append(cur)
    return None

def longest_valid_cycle(nums):
    n = len(nums)
    best = 0
    for i in cycle_nodes(nums):
        cur, L = i, 0
        while True:
            cur = (cur + nums[cur]) % n
            L += 1
            if cur == i:
                break
        best = max(best, L)
    return best

def count_cycle_nodes(nums):
    return len(cycle_nodes(nums))

def any_cycle_ignore_sign(nums):
    """Biến thể 4: mọi chu trình độ dài > 1, không cần cùng dấu."""
    n = len(nums)
    for i in range(n):
        cur = i
        for L in range(1, n + 1):
            cur = (cur + nums[cur]) % n
            if cur == i:
                if L > 1:
                    return True
                break
    return False


def is_valid_cycle(nums, cyc):
    n = len(nums)
    if cyc is None or len(cyc) < 2 or len(set(cyc)) != len(cyc):
        return False
    same_sign = all(nums[c] * nums[cyc[0]] > 0 for c in cyc)
    follows = all((cyc[t] + nums[cyc[t]]) % n == cyc[(t + 1) % len(cyc)] for t in range(len(cyc)))
    return same_sign and follows


print(find_valid_cycle([2, -1, 1, 2, 2]), longest_valid_cycle([2, -1, 1, 2, 2]))
print(count_cycle_nodes([1, -1, 5, 1, 4]), any_cycle_ignore_sign([1, -1]), any_cycle_ignore_sign([2, 2]))

random.seed(15)
for _ in range(500):
    n = random.randint(1, 8)
    nums = [random.choice([v for v in range(-9, 10) if v != 0]) for _ in range(n)]
    cyc = find_valid_cycle(nums)
    assert (cyc is not None) == Solution().circularArrayLoop(list(nums))
    if cyc is not None:
        assert is_valid_cycle(nums, cyc)
        assert longest_valid_cycle(nums) >= len(cyc)
print("OK biến thể")

[0, 2, 3] 3
2 True False
OK biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đi `n` bước từ mỗi chỉ số | chuồng bồ câu: đi `n` bước ⇒ vào chu trình | `O(n^2)` | `O(1)` | `n` nhỏ, cần code cực đơn giản / làm oracle |
| 2. Tô màu đường đi | mỗi đỉnh xử lý một lần | `O(n)` | `O(n)` | an toàn, không phá input |
| 3. Floyd + đặt `0` | rùa-thỏ, đánh dấu tại chỗ | `O(n)` | `O(1)` | follow-up, phỏng vấn |
| 4. Kahn | bóc đỉnh bậc vào 0; còn dư = chu trình | `O(n)` | `O(n)` | khi quen đồ thị / thứ tự tô-pô |
| 5. Union-Find | gộp nhóm, gặp lại nhóm = chu trình | `O(n α(n))` | `O(n)` | code ngắn, hợp khi đã có DSU |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[(i + v) % 4 for i, v in enumerate([3, -1, 2, -5])]`, và `Solution().circularArrayLoop([2, 2])`, `Solution().circularArrayLoop([1, 1])`, `Solution().circularArrayLoop([-1, 1])`. Điền `ans1_next` (list) và `ans1_bool` (list ba giá trị bool theo thứ tự ba lời gọi).

**Bài 2 (tính tay).** Với `nums = [3, -2, 1, 1, -4, 2]` (`n = 6`): tính mảng `next`, vẽ các mũi tên, tìm mọi chu trình và nói chu trình nào hợp lệ. Điền `ans2_next` (list) và `ans2_valid` (bool).

**Bài 3 (điền TODO).** Hoàn thành `loop_todo(nums)` theo Cách 4 (Kahn).

In [34]:
ans1_next = None
ans1_bool = None
ans2_next = None
ans2_valid = None

def loop_todo(nums):
    # TODO 1: n = len(nums); out = [-1] * n; indeg = [0] * n
    # TODO 2: for i in range(n): j = (i + nums[i]) % n; nếu j != i và nums[i]*nums[j] > 0: out[i] = j; indeg[j] += 1
    # TODO 3: q = deque(các i có indeg[i] == 0); removed = 0
    # TODO 4: while q: u = q.popleft(); removed += 1; v = out[u]; nếu v != -1: giảm indeg[v]; nếu bằng 0 thì q.append(v)
    # TODO 5: return removed < n
    pass

# KHÔNG gọi loop_todo ở đây để Run All không lỗi.

**Bài 4 (tự viết lại).** Viết `loop_oracle(nums)` **theo đúng định nghĩa**, không xem lại mục 12: với mỗi `i`, đi theo `next` tối đa `n` bước để xem có quay về `i` không; nếu có, kiểm tra độ dài `> 1` và cùng dấu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `array_nesting_like(nums)` đếm **số chu trình hợp lệ khác nhau** (mỗi chu trình đếm một lần). (b) Viết `can_escape(nums)` cho **mảng không vòng**: từ chỉ số `0` nhảy theo `nums[i]` (có thể tiến/lùi), nếu vượt ra ngoài `[0, n-1]` thì "thoát"; trả về `True` nếu thoát được, `False` nếu rơi vào vòng lặp mãi (gợi ý: dùng tập đã thăm).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: tính `next`: `(0+3)%4=3`, `(1-1)%4=0`, `(2+2)%4=0`, `(3-5)%4=2`. Với `[2,2]` mỗi phần tử có `2 % 2 == 0` nên tự trỏ; với `[1,1]` thì `0 ⇄ 1`; `[-1,1]` có chu trình nhưng khác dấu.
- **Mức 2 (vừa):** Bài 2: `next = [3, 5, 3, 4, 0, 1]`; vẽ: `0→3→4→0` (dấu `+3, +1, -4`: khác dấu), `1→5→1` (dấu `-2, +2`: khác dấu), `2→3`. Hãy kiểm tra từng chu trình. Bài 3: bám sát Cách 4 ở mục 7.
- **Mức 3 (sâu):** Bài 4: dùng `for _ in range(n)` rồi so với `i`; lưu `path`. Bài 5(a): mỗi chu trình xác định bởi tập đỉnh; dùng `frozenset` hoặc lấy đỉnh nhỏ nhất làm đại diện. Bài 5(b): `while 0 <= i < n and i not in seen:` rồi `seen.add(i); i += nums[i]`; thoát nếu `i` ra ngoài.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [35]:
# Bài 1
ans1_next = [3, 0, 0, 2]
ans1_bool = [False, True, False]
assert ans1_next == [(i + v) % 4 for i, v in enumerate([3, -1, 2, -5])]
assert ans1_bool == [Solution().circularArrayLoop(list(a)) for a in ([2, 2], [1, 1], [-1, 1])]

# Bài 2: nums = [3, -2, 1, 1, -4, 2]
nums2 = [3, -2, 1, 1, -4, 2]
ans2_next = [3, 5, 3, 4, 0, 1]
assert ans2_next == [(i + v) % 6 for i, v in enumerate(nums2)]
print("next =", ans2_next)
print("chu trình:", sorted(cycle_nodes(nums2)))
ans2_valid = oracle(nums2)
print("có chu trình hợp lệ?", ans2_valid)

next = [3, 5, 3, 4, 0, 1]
chu trình: []
có chu trình hợp lệ? False


Kiểm tra kết quả của Bài 2: `next = [3, 5, 3, 4, 0, 1]`. Các mũi tên: `0→3→4→0` (chu trình độ dài 3, dấu `+3, +1, -4`: **khác dấu**, loại), `1→5→1` (chu trình độ dài 2, dấu `-2, +2`: **khác dấu**, loại), `2→3` chỉ là nhánh đổ vào. Không có chu trình cùng dấu ⇒ `False`.

In [36]:
# Bài 3
def loop_todo(nums):
    n = len(nums)
    out = [-1] * n
    indeg = [0] * n
    for i in range(n):
        j = (i + nums[i]) % n
        if j != i and nums[i] * nums[j] > 0:
            out[i] = j
            indeg[j] += 1
    q = deque(i for i in range(n) if indeg[i] == 0)
    removed = 0
    while q:
        u = q.popleft()
        removed += 1
        v = out[u]
        if v != -1:
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)
    return removed < n

# Bài 4
def loop_oracle(nums):
    n = len(nums)
    for i in range(n):
        cur, path = i, [i]
        for L in range(1, n + 1):
            cur = (cur + nums[cur]) % n
            if cur == i:
                if L > 1 and all(nums[p] * nums[i] > 0 for p in path):
                    return True
                break
            path.append(cur)
    return False

random.seed(19)
for _ in range(500):
    n = random.randint(1, 8)
    nums = [random.choice([v for v in range(-9, 10) if v != 0]) for _ in range(n)]
    assert loop_todo(nums) == loop_oracle(nums) == Solution().circularArrayLoop(list(nums))
assert ans2_valid == loop_oracle(nums2) == False
print("OK bài 1-4")

OK bài 1-4


In [37]:
# Bài 5(a): đếm số chu trình hợp lệ khác nhau
def array_nesting_like(nums):
    n = len(nums)
    cycles = set()
    for i in range(n):
        cur, path = i, [i]
        for L in range(1, n + 1):
            cur = (cur + nums[cur]) % n
            if cur == i:
                if L > 1 and all(nums[p] * nums[i] > 0 for p in path):
                    cycles.add(frozenset(path))
                break
            path.append(cur)
    return len(cycles)

assert array_nesting_like([2, -1, 1, 2, 2]) == 1
assert array_nesting_like([1, 1, 1, 1]) == 1            # một chu trình qua cả 4 đỉnh
assert array_nesting_like([2, 2, 2, 2]) == 2            # 0<->2 và 1<->3
assert array_nesting_like([-1, -2, -3, -4, -5, 6]) == 0

# Bài 5(b): mảng không vòng, nhảy từ 0; vượt ra ngoài là thoát
def can_escape(nums):
    n = len(nums)
    i, seen = 0, set()
    while 0 <= i < n:
        if i in seen:
            return False
        seen.add(i)
        i += nums[i]
    return True

assert can_escape([1, 1, 5]) is True       # 0->1->2->7 (ra ngoài)
assert can_escape([2, -1, 1]) is True      # 0->2->3 (ra ngoài)
assert can_escape([1, -1]) is False        # 0->1->0 lặp
print("OK bài 5")

OK bài 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Floyd + đánh dấu `0`, `O(1)` bộ nhớ phụ)

```python
class Solution:
    def circularArrayLoop(self, nums: List[int]) -> bool:
        n = len(nums)
        nxt = lambda i: (i + nums[i]) % n
        for i in range(n):
            if nums[i] == 0:
                continue
            slow, fast = i, nxt(i)
            while nums[slow] * nums[fast] > 0 and nums[slow] * nums[nxt(fast)] > 0:
                if slow == fast:
                    if slow != nxt(slow):      # độ dài > 1
                        return True
                    break
                slow, fast = nxt(slow), nxt(nxt(fast))
            j = i
            while nums[j] * nums[nxt(j)] > 0:  # đánh dấu đường đi cùng dấu là đã xét
                k = nxt(j)
                nums[j] = 0
                j = k
            nums[j] = 0
        return False
```

**Checklist**

- [ ] `next(i) = (i + nums[i]) % n` (Python lo luôn số âm và bước lớn)?
- [ ] Chu trình phải **cùng dấu** (tích `> 0`) và **độ dài > 1** (loại tự trỏ)?
- [ ] Khi đổi dấu thì **dừng** và **không** đánh dấu đỉnh khác dấu?
- [ ] Biết bản sửa input (`O(1)` bộ nhớ) khác bản dùng `state` (`O(n)` bộ nhớ, không phá input)?
- [ ] Đã thử `[2,-1,1,2,2]` (True), `[-1,-2,-3,-4,-5,6]` (False), `[1,-1,5,1,4]` (True), `[1,-1]` (False), `[2,2]` (False), `[1]` (False)?
- [ ] Skeleton đúng `class Solution` / `def circularArrayLoop(self, nums: List[int]) -> bool`?